# Extract a larger ORD pool — CPU only, no GPU quota

Every conditions run so far draws on one pool of 297,000 ORD reactions, of which 138,869
carry at least one condition field and 38,191 carry a temperature. That last number is the
binding one: Model 1 gained 7 points going from 57k to 147k ORD reactions, so 38k is very
likely still inside the region where data helps, and a single run on it would leave
"temperature is hard" and "the temperature subset is small" indistinguishable.

So this kernel triples the pool. `--pool-count 900000` at the observed 27.5% temperature
coverage should yield roughly 110,000 temperature-bearing rows — a second point on the curve
rather than a second opinion.

**No GPU.** This is protobuf parsing and RDKit canonicalization; the weekly GPU quota is the
scarce resource in this project and it is spent on B3 and T1 running at the same time.

The sample is stratified, seeded, and excludes every product already in
`data/v2_ord_eval_targets.json`, so the enlarged pool stays disjoint from Model 1's ORD test
set by construction — the same guarantee the 297k pool had.

Roles are split here too, so the output is ready to train on: `reactants_smiles` becomes the
substrates and `reagents` the rest, exactly as `build_conditions_roles.py` does locally.

**Runtime:** several hours. Kaggle CPU kernels allow 12.

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import multiprocessing
print("CPU cores:", multiprocessing.cpu_count())

pool_count = 900000
output_dir = "/kaggle/working/v2_ord_train_900k"

In [ ]:
# Downloads the ORD dump from Hugging Face on first use (~1.1 GB) and parses it. The eval
# targets are committed in the repo, so the exclusion works without any extra input.
!python scripts/build_train_data_ord.py \
    --pool-count {pool_count} \
    --output-dir "{output_dir}" \
    2>&1 | tail -20

In [ ]:
import json, glob

for path in sorted(glob.glob(f"{output_dir}/*.jsonl")):
    rows = sum(1 for _ in open(path))
    if "conditions" in path:
        temp = sum(1 for line in open(path)
                   if json.loads(line).get("temperature_celsius") not in (None, ""))
        print(f"{path}: {rows} rows, {temp} with a temperature ({temp / rows:.1%})")
    else:
        print(f"{path}: {rows} rows")

In [ ]:
# Same roles split as the 138k corpus, so the two are directly comparable and the output can
# be trained on without another pass. One MCS per (fragment, product) pair, parallelized.
for split in ("train", "val"):
    !python scripts/build_conditions_roles.py \
        --input "{output_dir}/conditions_{split}.jsonl" \
        --output "/kaggle/working/roles_900k/conditions_{split}.jsonl" \
        2>&1 | tail -2

In [ ]:
import json

# The temperature-only subset, built here so the next training run needs no local step.
for split in ("train", "val"):
    src = f"/kaggle/working/roles_900k/conditions_{split}.jsonl"
    dst = f"/kaggle/working/roles_900k_temp/conditions_{split}.jsonl"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    kept = 0
    with open(dst, "w") as out:
        for line in open(src):
            if json.loads(line).get("temperature_celsius") not in (None, ""):
                out.write(line)
                kept += 1
    print(f"{dst}: {kept} rows")